In [1]:
# Cell 2 — Imports + helpers
from __future__ import annotations
import re
from dataclasses import dataclass
from typing import List, Optional, Tuple, Any, Dict

import fitz  # PyMuPDF


def normalize_ws(s: str) -> str:
    return re.sub(r"\s+", " ", s or "").strip()


def norm_token(w: str) -> str:
    w = (w or "").lower().strip()
    w = re.sub(r"^[^\w]+|[^\w]+$", "", w)
    return w

In [2]:
# Cell 3 — Robust anchor search (word-based, survives line breaks)

@dataclass
class AnchorMatch:
    page: int          # 1-based
    y0: float
    y1: float
    x0: float
    x1: float
    start_word_i: int
    end_word_i: int


def find_anchor_in_page_words(page: Any, anchor: str) -> List[AnchorMatch]:
    anchor_tokens = [norm_token(t) for t in normalize_ws(anchor).split()]
    anchor_tokens = [t for t in anchor_tokens if t]
    if len(anchor_tokens) < 3:
        return []

    words = page.get_text("words") or []
    words.sort(key=lambda w: (w[5], w[6], w[7], w[1], w[0]))  # reading-ish order
    page_tokens = [norm_token(w[4]) for w in words]

    n = len(anchor_tokens)
    if len(page_tokens) < n:
        return []

    matches: List[AnchorMatch] = []
    for i in range(0, len(page_tokens) - n + 1):
        if page_tokens[i:i+n] == anchor_tokens:
            xs0 = min(words[j][0] for j in range(i, i+n))
            ys0 = min(words[j][1] for j in range(i, i+n))
            xs1 = max(words[j][2] for j in range(i, i+n))
            ys1 = max(words[j][3] for j in range(i, i+n))
            matches.append(
                AnchorMatch(
                    page=page.number + 1,
                    y0=float(ys0), y1=float(ys1),
                    x0=float(xs0), x1=float(xs1),
                    start_word_i=i, end_word_i=i+n-1,
                )
            )
    return matches


def find_anchor_in_pdf(pdf_path: str, anchor: str, max_matches: int = 3) -> List[AnchorMatch]:
    doc = fitz.open(pdf_path)
    found: List[AnchorMatch] = []
    for pno in range(len(doc)):
        hits = find_anchor_in_page_words(doc.load_page(pno), anchor)
        if hits:
            found.extend(hits)
            if len(found) >= max_matches:
                break
    found.sort(key=lambda m: (m.page, m.y0, m.x0))
    return found[:max_matches]

In [3]:
# Cell 4 — NEW: Body-font estimation + strict heading detection + merging

@dataclass
class Heading:
    text: str
    page: int      # 1-based
    y0: float
    level: int     # 1 = highest
    font_size: float
    is_numbered: bool


HEADING_KEYWORDS = {
    "abstract", "introduction", "background", "related work", "methodology", "methods",
    "results", "discussion", "conclusion", "conclusions", "future work", "references",
    "acknowledgment", "acknowledgements", "appendix"
}

METADATA_BADWORDS = {
    "issn", "doi", "volume", "issue", "pages", "website", "journal", "http", "https", "www."
}


def estimate_body_font_size(pdf_path: str, max_pages: int = 12) -> float:
    """
    Estimate dominant body font size (weighted by characters) from first N pages.
    """
    doc = fitz.open(pdf_path)
    size_weight: Dict[float, int] = {}

    n_pages = min(len(doc), max_pages)
    for pno in range(n_pages):
        page = doc.load_page(pno)
        d = page.get_text("dict")
        for b in d.get("blocks", []):
            if b.get("type") != 0:
                continue
            for line in b.get("lines", []):
                for span in line.get("spans", []):
                    txt = span.get("text", "")
                    if not txt.strip():
                        continue
                    sz = round(float(span.get("size", 0.0)), 1)
                    size_weight[sz] = size_weight.get(sz, 0) + len(txt)

    if not size_weight:
        return 12.0

    # Most weighted size = body size
    body_size = max(size_weight.items(), key=lambda kv: kv[1])[0]
    return float(body_size)


def looks_like_metadata(line: str) -> bool:
    s = (line or "").lower()
    if any(w in s for w in METADATA_BADWORDS):
        return True
    # typical “||Volume||Issue||Pages||”
    if "||" in s and ("volume" in s or "issue" in s or "pages" in s):
        return True
    return False


def is_heading_candidate(text: str, avg_size: float, body_size: float, font_names: List[str]) -> bool:
    """
    Very strict: accept only if (a) matches heading patterns/keywords OR (b) clearly larger than body.
    """
    t = normalize_ws(text)
    if not t:
        return False
    low = t.lower()

    if looks_like_metadata(t):
        return False

    # very long lines are rarely headings
    if len(t) > 140:
        return False

    # numbered headings like "1", "1.2", "2.3.1 Title"
    if re.match(r"^(\d+(\.\d+)*)\s+\S+", t):
        return True

    # single-word / short known headings (Abstract, References, ...)
    if low in HEADING_KEYWORDS:
        return True
    if len(t.split()) <= 4 and any(k == low for k in HEADING_KEYWORDS):
        return True

    # Bold-ish font name as a weak signal (don’t rely on flags)
    is_boldish = any("bold" in (fn or "").lower() for fn in font_names)

    # Size rule: must be clearly bigger than body
    # (Body often 11–12; headings often 13–18.)
    if avg_size >= body_size + 1.6:
        return True
    if is_boldish and avg_size >= body_size + 0.8 and len(t.split()) <= 14:
        return True

    return False


def merge_multiline_headings(headings: List[Heading], y_gap: float = 3.5) -> List[Heading]:
    """
    Merge consecutive headings on same page with similar font size/level and very small vertical gaps.
    Useful for titles split across multiple lines.
    """
    if not headings:
        return headings

    merged: List[Heading] = []
    cur = headings[0]

    for h in headings[1:]:
        same_page = (h.page == cur.page)
        close_y = same_page and abs(h.y0 - cur.y0) <= 50  # rough
        similar_size = abs(h.font_size - cur.font_size) <= 0.4
        same_level = h.level == cur.level

        # If these look like a wrapped title line, merge them.
        if same_page and close_y and similar_size and same_level:
            cur = Heading(
                text=normalize_ws(cur.text + " " + h.text),
                page=cur.page,
                y0=min(cur.y0, h.y0),
                level=cur.level,
                font_size=cur.font_size,
                is_numbered=cur.is_numbered or h.is_numbered
            )
        else:
            merged.append(cur)
            cur = h

    merged.append(cur)
    return merged


def build_heading_index_strict(pdf_path: str, max_levels: int = 4) -> Tuple[List[Heading], float]:
    doc = fitz.open(pdf_path)
    body_size = estimate_body_font_size(pdf_path)

    candidates = []
    for pno in range(len(doc)):
        page = doc.load_page(pno)
        d = page.get_text("dict")
        for b in d.get("blocks", []):
            if b.get("type") != 0:
                continue
            for line in b.get("lines", []):
                spans = line.get("spans", [])
                if not spans:
                    continue
                line_text = "".join(sp.get("text", "") for sp in spans)
                t = normalize_ws(line_text)
                if not t:
                    continue

                avg_size = sum(float(sp.get("size", 0.0)) for sp in spans) / max(1, len(spans))
                fonts = [sp.get("font", "") for sp in spans]
                if not is_heading_candidate(t, avg_size, body_size, fonts):
                    continue

                # Exclude lines ending in comma/semicolon (often mid-sentence)
                if t.endswith((",", ";", ":")) and not re.match(r"^(\d+(\.\d+)*)\s+\S+", t):
                    continue

                y0 = min(sp["bbox"][1] for sp in spans if "bbox" in sp)
                is_num = bool(re.match(r"^(\d+(\.\d+)*)\s+\S+", t))
                candidates.append((t, pno + 1, float(y0), float(avg_size), is_num))

    if not candidates:
        return [], body_size

    # Level inference based on font sizes among *candidates* only
    sizes = sorted({round(c[3], 1) for c in candidates}, reverse=True)
    size_levels = sizes[:max_levels]

    def level_for(sz: float, is_numbered: bool, text: str) -> int:
        # If numbered, use numbering depth as a strong signal for hierarchy
        m = re.match(r"^(\d+(\.\d+)*)\s+", text)
        if m:
            depth = m.group(1).count(".") + 1  # 1, 2, 3...
            return max(1, min(4, depth))       # clamp
        # else map by size buckets
        s = round(sz, 1)
        nearest = min(size_levels, key=lambda x: abs(x - s))
        return size_levels.index(nearest) + 1

    headings = [
        Heading(text=t, page=pg, y0=y0,
                level=level_for(sz, is_num, t),
                font_size=sz,
                is_numbered=is_num)
        for (t, pg, y0, sz, is_num) in candidates
    ]
    headings.sort(key=lambda h: (h.page, h.y0))
    headings = merge_multiline_headings(headings)
    return headings, body_size

In [4]:
# Cell 5 — Extraction using STRICT headings (and a sanity fallback)

def find_heading_before_anchor(headings: List[Heading], loc: AnchorMatch) -> Optional[int]:
    best = None
    for i, h in enumerate(headings):
        if (h.page < loc.page) or (h.page == loc.page and h.y0 <= loc.y0):
            best = i
        else:
            break
    return best


def extract_between(doc: Any, start: Tuple[int, float], end: Optional[Tuple[int, float]] = None) -> str:
    start_page, start_y = start
    if end is None:
        end_page, end_y = len(doc), None
    else:
        end_page, end_y = end

    parts = []
    for pno in range(start_page, end_page + 1):
        page = doc.load_page(pno - 1)
        rect = page.rect
        y0 = start_y if pno == start_page else rect.y0
        y1 = end_y if (end_y is not None and pno == end_page) else rect.y1
        clip = fitz.Rect(rect.x0, y0, rect.x1, y1)
        parts.append(page.get_text("text", clip=clip))

    return "\n".join(parts).replace("\u00ad", "")


def extract_section_by_anchor_strict(pdf_path: str, anchor: str, headings: List[Heading]) -> Dict[str, Any]:
    doc = fitz.open(pdf_path)
    matches = find_anchor_in_pdf(pdf_path, anchor, max_matches=3)
    if not matches:
        return {"ok": False, "reason": "anchor_not_found", "anchor": anchor}

    loc = matches[0]
    h_idx = find_heading_before_anchor(headings, loc)

    # fallback: if no heading found, give a page-window around anchor
    if h_idx is None:
        page = doc.load_page(loc.page - 1)
        rect = page.rect
        top = max(rect.y0, loc.y0 - 300)
        bot = min(rect.y1, loc.y1 + 1200)
        txt = page.get_text("text", clip=fitz.Rect(rect.x0, top, rect.x1, bot)).replace("\u00ad", "")
        return {"ok": True, "method": "window_fallback", "section_title": None, "anchor_page": loc.page, "text": txt}

    h = headings[h_idx]

    # find end boundary: next heading with level <= current level
    end = None
    for nxt in headings[h_idx + 1:]:
        if nxt.level <= h.level:
            end = (nxt.page, nxt.y0)
            break

    text = extract_between(doc, (h.page, h.y0), end)

    # sanity: ensure the anchor is actually inside extracted text (normalized token check)
    norm_text = normalize_ws(text).lower()
    norm_anchor = normalize_ws(anchor).lower()
    if norm_anchor[:40] not in norm_text:
        # If not, widen: start one heading earlier (if exists)
        if h_idx > 0:
            h2 = headings[h_idx - 1]
            text2 = extract_between(doc, (h2.page, h2.y0), end)
            return {
                "ok": True, "method": "heading_bounds_widened",
                "section_title": h2.text, "anchor_page": loc.page, "text": text2
            }

    return {"ok": True, "method": "heading_bounds", "section_title": h.text, "anchor_page": loc.page, "text": text}

In [5]:
# Cell 6 — Run it on YOUR outputs (anchors) and print extracted full sections

PDF_PATH = r"static/test.pdf"  # <-- set this

anchors = [
    "Zero Trust Architecture (ZTA) represents a transformative approach to cybersecurity, shifting focus",
    "At the core of continuous verification lies the principle that every access attempt",
    "moves away from traditional perimeter-based defenses toward a model that assumes no implicit trust",
]

headings_strict, body_size = build_heading_index_strict(PDF_PATH)
print("Estimated body font size:", body_size)
print("Strict headings detected:", len(headings_strict))
print("First headings preview:")
for h in headings_strict[:20]:
    print(f"- p{h.page} lvl{h.level} size={h.font_size:.1f} :: {h.text}")

for a in anchors:
    sec = extract_section_by_anchor_strict(PDF_PATH, a, headings_strict)
    print("\n" + "="*80)
    print("Anchor:", a[:70], "...")
    print("Method:", sec.get("method"))
    print("Section title:", sec.get("section_title"))
    print("Anchor page:", sec.get("anchor_page"))
    txt = sec.get("text") or ""
    print("Extract length:", len(txt))
    print("\n--- first 1200 chars ---\n")
    print(txt[:1200])
    print("\n--- last 400 chars ---\n")
    print(txt[-400:])

Estimated body font size: 12.0
Strict headings detected: 59
First headings preview:
- p1 lvl1 size=18.4 :: Comprehensive Governance Framework, Implementation Methodologies, and Future Security Trends for Enterprise Environments
- p1 lvl2 size=12.1 :: Abstract
- p1 lvl1 size=10.4 :: 1 Unknown Author, More instructions how to create the bibtex entry.
- p4 lvl2 size=11.9 :: 2.2 Core Principles and Models
- p4 lvl3 size=10.6 :: 2.2.1 Least Privilege and Micro-Segmentation
- p5 lvl3 size=10.6 :: 2.2.2 Continuous Verification and Adaptive Access
- p6 lvl3 size=10.6 :: 2.2.3 Zero Trust Versus Traditional Security Models
- p7 lvl2 size=11.9 :: 2.3 Zero Trust in the Context of Cybersecurity Governance
- p7 lvl3 size=10.6 :: 2.3.1 Governance Models for Enterprise Security
- p8 lvl3 size=10.6 :: 2.3.2 Risk Management and Business Alignment
- p11 lvl2 size=11.9 :: 3.2 Cyber Risk as a Component of Enterprise Risk
- p12 lvl2 size=11.9 :: 3.3 Alignment of Security Objectives with Business Goals
- p12